# Prompt Engineering Lab
Author: Akansha Verma

# Part 1: Starting Simple - Create Initial Prompts
## 0. Setup


In [38]:
import os
from openai import OpenAI
import json
from collections import Counter
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

✅ Setup complete! OpenAI client initialized.


## Step 2: Create Initial Prompts


In [39]:
def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [40]:
# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""
 
# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
 
# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)

# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Sentiment Analysis Result:
The customer message can be classified as "Positive Feedback" or "Product Praise."
Product Description Result:
**Product Name:** SwiftClick Wireless Mouse

**Price:** $29.99

**Product Description:**

Experience unparalleled freedom and precision with the SwiftClick Wireless Mouse, your ultimate companion for seamless navigation and enhanced productivity. Designed for both work and play, this sleek and stylish mouse offers cutting-edge technology at an unbeatable price.

**Key Features:**

- **Wireless Convenience:** Say goodbye to tangled cords! The SwiftClick uses advanced 2.4 GHz wireless technology for a stable connection up to 33 feet away, allowing you to work or game without restrictions.

- **Ergonomic Design:** Crafted with comfort in mind, the SwiftClick features a contoured shape that fits perfectly in your hand, reducing strain during long hours of use. The textured grip ensures optimal control, making every click effortless.

- **High Precision T

# Part 2: Diagnosing Failures - Systematic Testing


In [41]:
def test_prompt(prompt, iterations=5):
    """
    Runs a prompt multiple times to check for consistency.
    """
    results = []
    for i in range(iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        results.append(response.choices[0].message.content.strip())
    
    # Print results for analysis
    for idx, res in enumerate(results):
        print(f"Run {idx+1}: {res}\n")
    return results

def analyze(prompt, n=15, label=""):
    results = test_prompt(prompt, n)
    print(f"\nRunning {n} times...")
    print("📊 Analysis:")
    print(f"📊 {label}: {len(set(results))}/{n} unique, avg {sum(len(r) for r in results)/len(results):.0f} chars")
    return results

## Step 3: Run Prompts 5 Times

In [42]:

# Run the same prompt 5 times with temperature=0.7
num_runs=5
print("=== TESTING SENTIMENT PROMPT ===")
analyze(sentiment_prompt_v1,num_runs,"Sentiment v1")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
analyze(product_prompt_v1,num_runs,"Product Description v1")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
analyze(extraction_prompt_v1,num_runs,"DATA EXTRACTION v1")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===
Run 1: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 2: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 3: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 4: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 5: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."


Running 5 times...
📊 Analysis:
📊 Sentiment v1: 2/5 unique, avg 92 chars
--------------------------------------------------------------------------------
=== TESTING PRODUCT DESCRIPTION PROMPT ===
Run 1: **Product Description: Wireless Precision Mouse - $29.99**

Elevate your computing experience with our Wireless Precision Mouse, designed for seamless navigation and unmatched comfort. Priced at just $29.99, this sleek and stylish mouse combines functionality 

## Analysis report for running prompt 5 times
### Are responses consistent in format?
No. 
- Sentiment: The model switches between (**Positive Feedback**) and quotation marks ("Positive Feedback")
- Product Description: While they all default to a paragraph-and-bullet format, the hierarchy, layout structure, and header styles vary.
- Data Extraction: The bullet points fluctuate between placing colons inside the bolding (**Order Number:**), outside the bolding (**Order Number**: ), or using no bolding at all. The extraction prompt kept a consistent bullet-list format but varied the field labels used ("Order Number" vs. "Item Ordered" vs. "Order Item Number").

### Are responses consistent in content?
- Sentiment: Yes, highly consistent — every run classified the message as positive, just with slightly different secondary label wording.
- Product description: No — the content is completely inconsistent. Each run invented different specifics (battery life claims, wireless range, feature lists like "quiet click technology" or "DPI adjustment" appearing in some runs and not others). Because the prompt had no specifications, the model hallucinated different brand names.
- Extraction: Yes, highly consistent — the actual values (#12345, March 15th, Fast delivery, Damaged packaging) were identical across all 5 runs. Only the labels naming those values changed.

### What types of variations do you see?
2 distinct categories:
1. Cosmetic/surface variation — different wording, bold vs. quoted text, different lead-in phrases ("Here is" vs. "Here's"), different field labels. This showed up in all three prompts.
2. Substantive/content variation — actual facts or details changing between runs. This was essentially absent in the sentiment and extraction prompts (the core answer never changed) but significant in the product description prompt, where the model generated different invented specs and even different product names each time.

## Step 4: Run Prompts 10 Times

In [43]:
# Run the same prompt 10 times with temperature=0.7
num_runs=10
print("=== TESTING SENTIMENT PROMPT ===")
analyze(sentiment_prompt_v1,num_runs,"Sentiment v1")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
analyze(product_prompt_v1,num_runs,"Product Description v1")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
analyze(extraction_prompt_v1,num_runs,"DATA EXTRACTION v1")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===
Run 1: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 2: The customer message can be classified as **Positive Feedback**.

Run 3: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 4: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 5: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 6: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 7: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 8: This customer message can be classified as **positive feedback** or **customer satisfaction**.

Run 9: The customer message can be classified as **Positive Feedback** or **Satisfaction**.

Run 10: This customer message can be classified as positive feedback.


Running 10 times...

## After running prompts 10 times
The responses are inconsistent in formatting, content and varied in syntax/cosmetics, content. 
A new failure pattern in the sentiment prompt — Run 4 dropped the "Customer Satisfaction" label outright, which never happened in the 5-run test. This confirms the expected outcome: more runs surfaced a rare failure mode that a 5-run sample didn't catch.

## Step 5: Run Prompts 15 Times and Create Failure Analysis

In [44]:
# Run the same prompt 15 times with temperature=0.7
num_runs=15
print("=== TESTING SENTIMENT PROMPT ===")
analyze(sentiment_prompt_v1,num_runs,"Sentiment v1")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
analyze(product_prompt_v1,num_runs,"Product Description v1")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
analyze(extraction_prompt_v1,num_runs,"DATA EXTRACTION v1")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===
Run 1: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 2: Positive feedback

Run 3: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 4: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 5: This customer message can be classified as **Positive Feedback / Satisfied Customer**.

Run 6: This customer message can be classified as **Positive Feedback**.

Run 7: The customer message can be classified as **Positive Feedback**.

Run 8: This customer message can be classified as **Positive Feedback**.

Run 9: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 10: This customer message can be classified as **Positive Feedback**.

Run 11: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 12: This customer message can be classified

## Failure Analysis - 15-Run Test (All Three Prompts)

## Summary Table

| Prompt | Unique Responses | Consistency (exact-match) | Avg. Length | Content Reliability |
|---|---|---|---|---|
| Sentiment classification | 9 / 15  | 43% - only 3 runs matched another run verbatim | 85 chars | Medium (core label drifts on ~20% of runs) |
| Product description | 15 / 15 (100% unique) | 0% - zero exact matches | 1,960 chars | Low (specs and identity change every run) |
| Data extraction | 11 / 15 | 29% - only 3 runs matched another verbatim | 157 chars | High(correct facts in 14/15 runs, 1 schema failure) |


## 1. Sentiment Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Lexical drift — model swaps in a different (but semantically similar) second label instead of the "expected" one | 3/15 (Runs 4, 5, 8) | "positive feedback or a **customer compliment**"; "...a **satisfied customer review**"; "...a **positive review**" |
| 2 | Single-label collapse — model drops the secondary label entirely | 2/15 (Runs 10, 12) | "classified as **Positive Feedback**." (no second label) |
| 3 | Cosmetic formatting drift — bold vs. quoted labels, "This" vs. "The" as sentence opener | ~10/15 | Recurs throughout |
| 4 | Casing inconsistency | 2/15 (Runs 3, 7... lowercase "positive feedback") | "**positive feedback** or **customer satisfaction**" |

**Root cause:** No fixed label was specified in the prompt, so the model is free to generate its own secondary label each time.


## 2. Product Description Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Inconsistent product naming/branding | 15/15 (name changes almost every run) | "SwiftClick," "SwiftConnect," "Wireless Precision Mouse," "Wireless Ergonomic Mouse," "Wireless Comfort Mouse," "Wireless Freedom Mouse" |
| 2 | Contradictory technical specs across runs | High | DPI ranges vary: 800/1200/1600 (most runs) vs. up to 2400 (Run 12) vs. up to 3200 (10-run test, Run 6) |
| 3 | Contradictory mechanics | Moderate | Most runs say "single AA battery"; Run 6 says "single AA battery" but frames it as long-lasting rechargeable-style claims elsewhere; earlier 10-run test had explicit "rechargeable battery" contradicting AA claims |
| 4 | Structural inconsistency | Moderate | Most runs use header + bullet "Key Features" format; Run 6 uses a hybrid paragraph-then-bullet-summary structure not seen elsewhere |
| 5 | Fabricated features not grounded in the prompt | High | "quiet click technology," "adjustable DPI," "sleep mode," "33 feet range" — none specified in the original prompt, all invented, and inconsistently applied |

**Root cause:** The prompt supplies only two constraints (product = wireless mouse, price = $29.99). Every other detail — name, features, specs, structure — is generated freely, so the model has no anchor to stay consistent against. This is the only prompt with 0% exact-match consistency across all 15 runs.

---

## 3. Data Extraction Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Field-name inconsistency | ~5/15 | "Order Number" (most common) vs. "Order Item Number" (Run 5) vs. "Item Ordered" (Run 8) |
| 2 | Field renamed to a shorter variant | 3/15 (Runs 3, 6, 10) | "Delivery Speed" → "Delivery" |
| 3 | Punctuation inconsistency on the ID value | 1/15 (Run 14) | "#12345" → "12345" (# dropped) |
| 4 | Schema restructuring (most severe failure) | 1/15 (Run 10) | Header changed to "Customer Feedback Summary:"; "Packaging Condition" field was dropped and replaced with a merged "**Issue**: Damaged packaging" field |
| 5 | Preamble sentence variation | ~8/15 | "Here is the extracted information...", "Here are the extracted details...", or no preamble at all (direct bullet list) |

**Root cause:** No output schema (e.g., JSON with fixed keys) was specified, so while the model reliably extracts the *correct facts* (14/15 runs got all four values right), it's free to invent its own field names and structure each time. Run 10 is the standout failure: it didn't just relabel a field, it changed what fields existed — this would silently break any code parsing by key name.

---

## Cross-Prompt Failure

| Failure Type | Sentiment | Product Description | Extraction |
|---|---|---|---|
| Cosmetic wording/formatting drift | Yes (dominant) | Yes | Yes (dominant) |
| Content/fact drift | Yes (minor, 2 patterns) | Yes (severe, pervasive) | No (facts stayed correct) |
| Schema/structure drift | No | Yes (structure mostly stable, specs unstable) | Yes (1 severe case) |
| Fabrication / hallucination | No | Yes (high — invented specs) | No |
| Exact-match consistency | 20% | 0% | 20% |

## Overall Findings at 15 Runs

1. **More runs surfaced failure modes invisible at smaller sample sizes.** The sentiment prompt's "single-label collapse" (Runs 10, 12) never appeared in the 5-run or 10-run tests. The extraction prompt's schema-breaking Run 10 was also a first — a genuinely new, more severe failure category, not just more of the same wording noise.
2. **Consistency does not equal correctness, and vice versa.** The extraction prompt has the same exact-match rate as the sentiment prompt (12/15 unique) but is far more reliable at the fact level — because its answer space (facts in the source text) is narrow, while sentiment's secondary-label space is unconstrained.
3. **The product description prompt is the clear outlier** — 0% exact-match consistency across all 15 runs, and the only prompt where the model contradicts itself on factual-sounding claims (specs, battery type) between runs. This is a hallucination risk, not just a style-variation risk.
4. **Failure severity ranks:** Product description (fabricated, contradictory specs) > Sentiment (occasional label collapse/substitution) > Extraction (correct facts, but unstable field-naming that would break automated parsing).



# Part 3: Iteration 1 - Rewriting Simple Prompts
Objective: Improve prompts based on failure analysis by adding clarity, format requirements, and constraints.

### Step 6: Improve Sentiment Analysis Prompt

In [45]:
sentiment_prompt_v2 = """
Please analyze the sentiment of this customer's message "I love this product! It's exactly what I needed."
FORMAT:
1. You must classify the sentiment in one of the following terms: POSITIVE, NEGATIVE, or NEUTRAL.
2. Do not include any conversational filler, introductory text, markdown formatting, or punctuation.
CONSTRAINT: Use ONLY one of the classification word
"""
# Run the same prompt 15 times with temperature=0.7
num_runs=15
print("=== TESTING SENTIMENT PROMPT ===")
analyze(sentiment_prompt_v2,num_runs,"Sentiment v2")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===
Run 1: POSITIVE

Run 2: POSITIVE

Run 3: POSITIVE

Run 4: POSITIVE

Run 5: POSITIVE

Run 6: POSITIVE

Run 7: POSITIVE

Run 8: POSITIVE

Run 9: POSITIVE

Run 10: POSITIVE

Run 11: POSITIVE

Run 12: POSITIVE

Run 13: POSITIVE

Run 14: POSITIVE

Run 15: POSITIVE


Running 15 times...
📊 Analysis:
📊 Sentiment v2: 1/15 unique, avg 8 chars
--------------------------------------------------------------------------------


### Step 7: Improve Product Description Prompt

In [46]:
product_prompt_v2 = """
Write compelling product description for a wireless mouse that costs $29.99
Structure the descriptions as:
1. Use the title: **Tech Flow Wireless Mouse**
2. A short (3 lines) introductory paragraph
3. Highlight **Key Features** exactly titled and write 3 bullet points of features
Lenght Contraint: The entire product description must be under 120 words.
Style guideline: 
- Tone: Professional
- Keep it concise
- Do not exaggerate with phrases such as "greatest mouse ever"
- Do not use emoji
"""
num_runs=15
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
analyze(product_prompt_v2,num_runs,"Product Description v2")
print("-" * 80)

=== TESTING PRODUCT DESCRIPTION PROMPT ===
Run 1: **Tech Flow Wireless Mouse**

Experience seamless navigation with the Tech Flow Wireless Mouse, designed for professionals and everyday users alike. Its ergonomic shape ensures comfort during extended use, while its wireless connectivity offers freedom from clutter. Elevate your productivity with this reliable accessory.

**Key Features**  
- **Ergonomic Design:** Contoured shape for maximum comfort during long hours of use.  
- **Long-Lasting Battery:** Enjoy up to 12 months of use on a single AA battery, reducing downtime.  
- **Advanced Optical Sensor:** Precise tracking on various surfaces for a smooth user experience.

Run 2: **Tech Flow Wireless Mouse**  
Experience seamless navigation with the Tech Flow Wireless Mouse, designed for both productivity and comfort. Its ergonomic design and advanced technology ensure smooth operation whether you’re at your desk or on the go.

**Key Features**  
- **Ergonomic Design**: Fits comfortabl

### Step 8: Improve Data Extraction Prompt

In [47]:
extraction_prompt_v2="""
Extract relevant details from this customer feedback: 
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
1. OUTPUT FORMAT: JSON object
2. Include following fields in the output:
- "order_number" (include # symbol)
- "order_date"
- "delivery_speed"
- "package_condition"
3. Do not use any conversational text, introductory remarks, or explanations.
"""
num_runs
print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
analyze(extraction_prompt_v2,num_runs,"Data Extraction v2")
print("-" * 80)



=== TESTING DATA EXTRACTION PROMPT ===
Run 1: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 2: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 3: ```json
{
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}
```

Run 4: ```json
{
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}
```

Run 5: ```json
{
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}
```

Run 6: ```json
{
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}
```

Run 7: ```json
{
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition":

## Comparision with Version 1

| Prompt | v1 Consistency | v2 Consistency | What fixed | What still needs work |
|---|---|---|---|---|
| Sentiment classification | 20% (12/15 unique) | 100% (1/15 unique) | Perfect Output | -- |
| Product description | 0% (15/15 unique) | High Structural Consistency | Branding, format, length, and style are locked in | Still hallucinating technical specs |
| Data extraction | 20% (12/15 unique) | 93% (2/15 unique) | Slight reduction in variations. Value extraction (e.g., #12345) was slightly more stable | Worked with structure |

# Part 4: Iteration 2 - Adding Structure & Constraints
Objective: Add few-shot examples, Chain-of-Thought reasoning, and advanced structure to dramatically improve consistency.
### Step 9: Add Few-Shot Examples to Sentiment Analysis

In [48]:
sentiment_prompt_v3="""
Classify the sentiment of customer message. Use exactly one of these classifiers: 
POSITIVE, NEGATIVE, or NEUTRAL.
Review these examples for the exact output format:
Message: "The delivery took two weeks and the item arrived broken."
Sentiment: NEGATIVE
Message: "It is a standard USB cable. It works as expected, nothing special."
Sentiment: NEUTRAL
Message: "{message}"
Sentiment:
"""
num_runs=15
print("=== TESTING SENTIMENT PROMPT ===")
analyze(sentiment_prompt_v3.format(message="I love this product! It's exactly what I needed."),num_runs,"Sentiment v3")
print("-" * 80)


=== TESTING SENTIMENT PROMPT ===
Run 1: Sentiment: POSITIVE

Run 2: Sentiment: POSITIVE

Run 3: POSITIVE

Run 4: Sentiment: POSITIVE

Run 5: Sentiment: POSITIVE

Run 6: POSITIVE

Run 7: Sentiment: POSITIVE

Run 8: Sentiment: POSITIVE

Run 9: Sentiment: POSITIVE

Run 10: POSITIVE

Run 11: Sentiment: POSITIVE

Run 12: POSITIVE

Run 13: POSITIVE

Run 14: Sentiment: POSITIVE

Run 15: Sentiment: POSITIVE


Running 15 times...
📊 Analysis:
📊 Sentiment v3: 2/15 unique, avg 15 chars
--------------------------------------------------------------------------------


### Step 10: Add Chain-of-Thought to Data Extraction


In [49]:
extraction_prompt_v3="""
Extract relevant details from the customer message: 
"{message}"
Step 1: Thinking Process
Briefly reason before answer the following:
1. What is the exact order number in the message (including any # symbol)?
2. What is the exact order date in the message?
3. Is the delivery speed mentioned in the message? If yes, then how is it described in the text (e.g. fast, slow, delayed, on-time)?
If no delivery speed is mentioned, then use null as return value
4. How is the packaging condition described in the message (e.g. damaged, intact, ok)? 
If nothing is mentioned about packaging, then use null as return value
Step 2: Output only a JSON object on its own, with no conversational text, introductory remarks, or explanations.
Use exactly this schema and these four keys, no more, no fewer:
{{
  "order_number": "",
  "order_date": "",
  "delivery_speed": "",
  "package_condition": ""
}}
Rules:
- For "order_number": include the "#" symbol
- "order_date": copy exactly as written in the message
- "delivery_speed": one or two words describing delivery speed, derived from reasoning
- "package_condition": one or two words describing packaging condition, derived from reasoning
- Do not repeat the reasoning inside the JSON. 
- Do not add any text after the JSON object.
"""
print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
analyze(extraction_prompt_v3.format(message="I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."),15,"Data Extraction v3")
print("-" * 80)



=== TESTING DATA EXTRACTION PROMPT ===
Run 1: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 2: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 3: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 4: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 5: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 6: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 7: {
  "order_number": "#12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "package_condition": "damaged"
}

Run 8: {
  "order_number": "#12345",
  "or

### Step 11: Add Few-Shot Examples and Structure to Product Description

In [50]:
product_prompt_v3 = """
Write a compelling product description for a {product} that costs ${price:.2f}.
Follow the exact structure and style shown in these examples:
EXAMPLE 1
Product: Mechanical keyboard, $49.99

**Wave Mechanical Keyboard**

Built for people who type for a living, the Wave Mechanical Keyboard pairs tactile precision with a compact frame. It fits any desk without sacrificing feel. Designed for long sessions without fatigue.

**Key Features**
- Hot-swappable mechanical switches for custom feel
- Compact 75% layout that saves desk space
- USB-C connection with a detachable braided cable

---
EXAMPLE 2
Product: Laptop stand, $34.99

**Riser Aluminum Laptop Stand**

The Riser lifts your laptop to eye level, easing neck strain during long work sessions. Machined from a single block of aluminum, it stays stable and looks refined. A simple upgrade for any desk setup.

**Key Features**
- Adjustable height with 6 locking positions
- Solid aluminum build rated for laptops up to 16"
- Foldable design for easy transport

NOW WRITE (same structure and tone as above):
Product: {product}, ${price:.2f}

Structure:
1. Title in bold, exactly in the format: **[Name] {product}**
2. A short introductory paragraph (exactly 3 lines)
3. A section titled exactly **Key Features** with exactly 3 bullet points

Length constraint: The entire product description must be under 120 words total.

Style guideline:
- Tone: Professional
- Keep it concise
- Do not exaggerate with phrases such as "greatest mouse ever"
- Do not use emoji
- Do not invent numeric specs (DPI, battery months, range in feet) unless they are standard, uncontroversial claims for this product category
"""
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
analyze(product_prompt_v3.format(product="wireless mouse", price=29.99),15,"Product Description v3")
print("-" * 80)


=== TESTING PRODUCT DESCRIPTION PROMPT ===
Run 1: **ErgoFlow Wireless Mouse**

The ErgoFlow Wireless Mouse combines ergonomic design with seamless performance, ensuring comfort and efficiency for daily tasks. Its sleek, lightweight structure fits naturally in your hand, making it ideal for extended use. A reliable companion for both work and play.

**Key Features**
- Ergonomic shape designed for long-lasting comfort  
- Reliable wireless connection with a range of up to 33 feet  
- Long-lasting battery life with an energy-saving mode

Run 2: **ErgoFlex Wireless Mouse**

The ErgoFlex Wireless Mouse combines comfort and precision for an enhanced computing experience. Its ergonomic design fits naturally in your hand, allowing for extended use without fatigue. A reliable companion for both work and play.

**Key Features**
- Ergonomic shape designed to reduce wrist strain
- Wireless connectivity with a reliable range for seamless use
- Long-lasting battery life with quick recharge capabilit

### Compare all version 3 results with version 1

| Task Type | V1 Baseline Consistency | V2 Iteration Consistency | V3 Few-Shot/CoT Consistency | Core Issue & Resolution |
| :--- | :--- | :--- | :--- | :--- |
| **Sentiment Analysis** <br>*(Classification)* | **20%** <br>(12/15 unique) | **100%** <br>(1/15 unique) | **93%** <br>(2/15 unique) | Fixed taxonomy variations by switching from descriptive text to a strict string enum template. |
| **Data Extraction** <br>*(Structured JSON)* | **20%** <br>(12/15 unique) | **93%** <br>(2/15 unique) | **100%** <br>(1/15 unique) | -- |
| **Product Description** <br>*(Creative/Generation)* | **0%** <br>(15/15 unique) | **0%** <br>(15/15 unique) | **0%** <br>(15/15 unique) | Full-text consistency never improved — every version produced 15/15 unique text. But this metric hides real gains: v2 locked the product title to "Tech Flow Wireless Mouse" in 15/15 runs (a fixed-name instruction, no few-shot needed) and cut average length from 1,954 → 644 chars. v3 (few-shot, flexible title) cut length further to 480 chars and suppressed numeric spec invention (no DPI values in any of 15 runs), but let the product name drift again (clustered around "Ergo-"/"Precision," not fixed). |


### Part 5: Tuning for Different Tasks
Objective: Adapt your prompts for variations of each task type and document what works best.

### Step 12: Create Task Variations

In [51]:
sentiment_variations = [
    "I love this product! It's exactly what I needed.",   
    "This is the worst purchase I've ever made. It broke after one day.",          
    "I received the item on Tuesday. It matches the description on the website.",       
    "It's okay I guess, does what it says but nothing more.",  
    "Oh great, ANOTHER broken product. Just what I needed this week.",   
]

print("=== TESTING: Sentiment prompt variations ===")
for msg in sentiment_variations:
    print(f"Input: {msg}")
    analyze(sentiment_prompt_v3.format(message=msg),15,"Sentiment variations")
    print("---")

=== TESTING: Sentiment prompt variations ===
Input: I love this product! It's exactly what I needed.
Run 1: Sentiment: POSITIVE

Run 2: POSITIVE

Run 3: POSITIVE

Run 4: Sentiment: POSITIVE

Run 5: POSITIVE

Run 6: Sentiment: POSITIVE

Run 7: POSITIVE

Run 8: POSITIVE

Run 9: Sentiment: POSITIVE

Run 10: POSITIVE

Run 11: POSITIVE

Run 12: Sentiment: POSITIVE

Run 13: Sentiment: POSITIVE

Run 14: Sentiment: POSITIVE

Run 15: POSITIVE


Running 15 times...
📊 Analysis:
📊 Sentiment variations: 2/15 unique, avg 13 chars
---
Input: This is the worst purchase I've ever made. It broke after one day.
Run 1: Sentiment: NEGATIVE

Run 2: NEGATIVE

Run 3: NEGATIVE

Run 4: NEGATIVE

Run 5: NEGATIVE

Run 6: NEGATIVE

Run 7: NEGATIVE

Run 8: NEGATIVE

Run 9: NEGATIVE

Run 10: NEGATIVE

Run 11: NEGATIVE

Run 12: Sentiment: NEGATIVE

Run 13: Sentiment: NEGATIVE

Run 14: Sentiment: NEGATIVE

Run 15: NEGATIVE


Running 15 times...
📊 Analysis:
📊 Sentiment variations: 2/15 unique, avg 11 chars
---
Input: I

In [52]:
product_variations = {
    "higher_price": ("Wireless mouse", 79.99),
    "different_product": ("Bluetooth keyboard", 39.99),
    "budget_tier": ("Wired mouse", 12.99),
}
print("=== TESTING: Product description variations ===")
for name, (product, price) in product_variations.items():
    print(f"\nVariation: {name} ({product}, ${price})")
    analyze(product_prompt_v3.format(product=product, price=price),15,"Product Description variations")
    print("-" * 80)

=== TESTING: Product description variations ===

Variation: higher_price (Wireless mouse, $79.99)
Run 1: **Precision Wireless Mouse**

The Precision Wireless Mouse is engineered for both work and play, delivering seamless connectivity and ergonomic comfort. Its sleek design fits naturally in your hand, ensuring hours of productivity without discomfort. Experience the freedom of wireless technology without compromise.

**Key Features**
- Advanced optical sensor for accurate tracking on various surfaces
- Ergonomic shape designed to reduce wrist strain
- Long-lasting battery life with quick charging capabilities

Run 2: **ErgoFlex Wireless Mouse**

Engineered for comfort and precision, the ErgoFlex Wireless Mouse provides an ergonomic grip that reduces wrist strain during extended use. Its sleek design complements any workspace while delivering reliable performance. A must-have for professionals who prioritize productivity.

**Key Features**
- Ergonomic design with customizable side grip

In [53]:
extraction_variations = {
    "missing_field": "I ordered item #55432 on April 2nd. Still waiting for it to arrive.",  # no delivery speed or packaging mentioned
    "multiple_items": "I ordered items #111 and #222 on May 5th. #111 arrived fast, #222 was delayed and the box was damaged.",  # schema assumes one order
    "extra_noise": "Hi there! Hope you're doing well. So I ordered #78901 back on June 10th, and I just wanted to say the delivery was super quick, though the packaging had a small dent in it. Thanks!",  # irrelevant conversational filler
    "different_domain": "Booking #BK4471 confirmed for June 20th. Check-in was smooth but the room wasn't ready on time.",  # tests if the schema breaks outside e-commerce
}
num_runs = 15
print("=== TESTING: Extraction variations ===")
for name, msg in extraction_variations.items():
    print(f"\n=== Variation: {name} ===")
    responses = test_prompt(extraction_prompt_v3.format(message=msg), num_runs)

    valid_json_count = 0
    for r in responses:
        try:
            json.loads(r)
            valid_json_count += 1
        except json.JSONDecodeError:
            pass

    print("📊 Analysis:")
    print(f"Input: {msg}")
    print(f"Valid JSON: {valid_json_count} / {num_runs}")
    print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
    print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
    print("-" * 80)

=== TESTING: Extraction variations ===

=== Variation: missing_field ===
Run 1: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 2: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 3: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 4: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "waiting",
  "package_condition": null
}

Run 5: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 6: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 7: {
  "order_number": "#55432",
  "order_date": "April 2nd",
  "delivery_speed": "delayed",
  "package_condition": null
}

Run 8: {
  "order_number": "#55

## Step 13: Final Evaluation and Comparison

**Scope:** Comparing v1 (baseline, zero-shot/minimal instruction) against v3 (final, tuned) prompts across all three tasks — sentiment classification, data extraction, product description generation. Each v3 prompt was run 15 times on its base case, then stress-tested on 4–5 task variations (Step 12) to check generalization. All numbers below come from actual test runs in this lab; no figures are estimated.


## 1. Summary

| Task | v1 Consistency | v3 Consistency (base case) | Verdict |
|---|---|---|---|
| Sentiment Analysis | 20% (label content itself drifted) | 93% (label never wrong; only a cosmetic prefix varies) | **Fixed** — semantic accuracy solved |
| Data Extraction | 20% (inconsistent field names, no schema) | 100% (fixed schema, valid JSON, correct facts every run) | **Fixed** — fully solved on base case, holds up well under variation |
| Product Description | 0% (text always unique, no structure, hallucinated specs) | 0% (text still always unique) but **structure 100% locked**, length cut ~75%, spec hallucination eliminated | **Partially fixed** — the right metric changed, not the raw uniqueness number |

The clearest lesson across all three tasks: **raw text-uniqueness is the wrong metric for creative tasks.** Sentiment and extraction have narrow, correct answers, so text-level consistency is a meaningful proxy for quality. Product description is meant to vary — the real wins there were structural compliance, length control, and hallucination suppression, none of which the uniqueness metric captures.


## 2. Sentiment Analysis

### Base Case (original message: "I love this product! It's exactly what I needed.")

| Version | Unique | Consistency | Failure Pattern |
|---|---|---|---|
| v1 | 12/15 | 20% | Secondary-label substitution (3 runs), single-label collapse (2 runs), casing/formatting drift |
| v3 | 2/15 | 93% | Only "POSITIVE" vs. "Sentiment: POSITIVE" prefix varies — the label itself never changed |

**What changed:** v3 replaced the model's free-form secondary-label generation with a fixed enum (`POSITIVE, NEGATIVE, NEUTRAL`) and few-shot examples anchoring the exact output format.

**Improvement metric:** the label itself never flipped within any of the 5 sets of 15 runs — **100% internal label stability across all tested inputs**, including a correctly-resolved sarcasm case. The only unresolved issue is the "Sentiment:" prefix formatting split, present in 4 of 5 variations at ~50/50.

**Known limitation:** one input type (factual/no-sentiment message) resolved consistently but arguably incorrectly to POSITIVE — the few-shot examples never demonstrated how to handle a truly neutral, sentiment-free message.

---

## 3. Data Extraction

### Base Case (original message: order #12345, March 15th, fast delivery, damaged packaging)

| Version | Unique | Consistency | Valid JSON | Failure Pattern |
|---|---|---|---|---|
| v1 | 12/15 | 20% | 0/15 (not JSON) | Field-name drift ("Order Number" vs "Item Ordered" vs "Order Item Number"), one full schema restructure |
| v3 (interim, buggy) | 7/15 | 57% | 15/15 (valid but wrong) | Missing source message → 100% well-formed but fully hallucinated JSON |
| v3 (final) | 1/15 | 100% | 15/15 | None — fully correct, fully consistent |

**What changed:** locked JSON schema with exactly 4 named keys, explicit "no text after the JSON object" rule, and (after fixing an interim bug) the source message correctly embedded in the prompt.

**Improvement metric:** JSON validity held at **100% (60/60)** across every variation — the schema-lock generalizes completely. Text consistency ranged from 86–100% depending on input complexity.

**Known limitation:** JSON validity does not equal semantic correctness. The `multiple_items` case is a real, unresolved schema design gap — the prompt needs an explicit rule for multi-order messages (e.g., return an array, or extract only the first mentioned order) before it can be called fully solved.

## 4. Product Description Generation

### Base Case ($29.99 wireless mouse)

| Version | Unique | Consistency | Avg. Length | Structure Locked? | Numeric Spec Fabrication |
|---|---|---|---|---|---|
| v1 | 15/15 | 0% | 1,954 chars | No — 1/15 runs broke format entirely | High (DPI 800–3200, contradictory battery claims) |
| v3 | 15/15 | 0% | 497 chars | Yes — 15/15 runs matched title/intro/bullets format | None observed (0 DPI values in 15 runs) |

**What changed:** two few-shot examples (unrelated products) plus an explicit structure spec, word limit, tone rules, and an instruction against inventing numeric specs.

**Improvement metric:** structural format (title/intro/Key Features/bullets) held **100% (60/60)** across every variation, and numeric spec fabrication stayed suppressed (no DPI values in any of 60 runs across all conditions). Average length stayed in the 460–510 character range regardless of product or price — the length constraint generalizes reliably.

**Known limitation:** raw text uniqueness never improved (still 0% by the strict formula, same as v1) — few-shot priming controls *style* of variation, not elimination of it. Naming stability turned out to be product/price-dependent rather than a fixed property of the prompt: strongest for the keyboard and budget mouse, weakest for the original and higher-priced wireless mouse (where two competing naming conventions — "Ergo-" and "Precision" — kept splitting the field). Price also introduced a new, mild fabrication pattern (battery type) not seen at the original price point.

